In [ ]:
from pathlib import Path
import numpy as np
import traceback

# 改成你的目录或文件列表
TARGET = Path("/work/SSR/share/data/skiing/skiing_unity_dataset/sam3d_body_results/inference/female/Anim_Female_Skier_Braking/frames/capture_L0_A010/000006_sam3d_body.npz")  # 可以是目录，也可以是单个 .npz 文件
RECURSIVE = True         # 目录下是否递归扫描

def find_npz_files(target: Path, recursive: bool = True):
    if target.is_file() and target.suffix.lower() == ".npz":
        return [target]
    if target.is_dir():
        pattern = "**/*.npz" if recursive else "*.npz"
        return sorted(target.glob(pattern))
    return []

def check_npz_file(npz_path: Path):
    """
    返回: (is_bad, err_msg)
    is_bad=True 表示文件损坏/不可读
    """
    try:
        with np.load(npz_path, allow_pickle=False) as data:
            # 强制读取每个数组，触发潜在 CRC/解压/截断错误
            for k in data.files:
                _ = data[k]
        return False, ""
    except Exception as e:
        return True, f"{type(e).__name__}: {e}"

def main():
    files = find_npz_files(TARGET, RECURSIVE)
    if not files:
        print("未找到 .npz 文件")
        return

    bad_files = []
    for f in files:
        is_bad, err = check_npz_file(f)
        if is_bad:
            bad_files.append((f, err))
            print(f"[BAD] {f}\n      {err}")
        else:
            print(f"[OK ] {f}")

    print("\n====== 检查完成 ======")
    print(f"总数: {len(files)}")
    print(f"坏文件: {len(bad_files)}")

    # 保存坏文件清单
    if bad_files:
        out = Path("bad_npz_report.txt")
        with out.open("w", encoding="utf-8") as fw:
            for f, err in bad_files:
                fw.write(f"{f}\t{err}\n")
        print(f"坏文件报告已保存: {out.resolve()}")

if __name__ == "__main__":
    main()